#Data quality checks

In [ ]:
import pandas as pd
import numpy as np

# Load the file
file_path = r"C:\....\bets.csv"
df = pd.read_csv(file_path)

# ---------------------------------------
# Data Quality Checks
# ---------------------------------------
df['dq_bet_id_null'] = df['bet_id'].isnull()
df['dq_bet_id_duplicate'] = df['bet_id'].duplicated(keep=False)
df['dq_customer_id_null'] = df['customer_id'].isnull()
df['dq_bet_datetime_invalid'] = df['bet_datetime'].isnull()
df['dq_invalid_betting_amount'] = df['betting_amount'] <= 0
df['dq_invalid_price'] = df['price'] <= 1
df['dq_invalid_stake_type'] = ~df['stake_type'].isin(['cash', 'bonus'])
df['dq_invalid_bet_result'] = ~df['bet_result'].isin(['return', 'no-return'])
df['dq_invalid_category'] = ~df['category'].isin(['sports', 'racing'])
df['dq_invalid_bet_num'] = df['bet_num'] < 1

# ---------------------------------------
# Expected payout logic
# ---------------------------------------
def expected_payout(row):
    if row['bet_result'] != 'return':
        return 0
    if row['stake_type'] == 'cash':
        return row['betting_amount'] * row['price']
    elif row['stake_type'] == 'bonus':
        return row['betting_amount'] * row['price'] - row['betting_amount']
    return np.nan

df['expected_payout'] = df.apply(expected_payout, axis=1)
df['dq_invalid_payout'] = (
    (df['bet_result'] == 'return') &
    (~np.isclose(df['payout'], df['expected_payout'], atol=0.01))
)

# ---------------------------------------
# rob_return_for_entain calculation
# ---------------------------------------
def calculate_return(row):
    if row['bet_result'] == 'no-return' and row['stake_type'] == 'cash':
        return row['betting_amount']
    elif row['bet_result'] == 'no-return' and row['stake_type'] == 'bonus':
        return 0
    elif row['bet_result'] == 'return' and row['stake_type'] == 'cash':
        return -(row['payout'] - row['betting_amount'])
    elif row['bet_result'] == 'return' and row['stake_type'] == 'bonus':
        return -row['payout']
    else:
        return None

df['rob_return_for_entain'] = df.apply(calculate_return, axis=1)

# ---------------------------------------
# Validate against return_for_entain column
# ---------------------------------------
df['dq_mismatch_return_for_entain'] = ~np.isclose(
    df['rob_return_for_entain'],
    df['return_for_entain'],
    atol=0.01,
    equal_nan=True
)

# Check if bet_num is sequential per customer (1, 2, ..., n)
def is_bet_num_valid(sub_df):
    expected = list(range(1, len(sub_df) + 1))
    actual = sub_df['bet_num'].tolist()
    return actual == expected

# Initialize all rows as valid
df['dq_nonsequential_bet_num'] = False

# Apply check per customer_id
for cust_id, group in df.groupby('customer_id'):
    group_sorted = group.sort_values(by='bet_num')
    if not is_bet_num_valid(group_sorted):
        df.loc[group_sorted.index, 'dq_nonsequential_bet_num'] = True

# ---------------------------------------
# Count issues and identify failing columns
# ---------------------------------------
dq_columns = [col for col in df.columns if col.startswith('dq_')]
df['dq_issues_count'] = df[dq_columns].sum(axis=1)

def get_issue_columns(row):
    return [col for col in dq_columns if row[col]]

df['dq_issue_columns'] = df.apply(get_issue_columns, axis=1)

# ---------------------------------------
# Summary print
# ---------------------------------------
print("=== Data Quality Issue Summary ===")
for col in dq_columns:
    print(f"{col}: {df[col].sum()} issue(s)")

# Sample problem rows
print("\n=== Sample Rows with Issues ===")
print(df[df['dq_issues_count'] > 0][['bet_id', 'customer_id', 'dq_issues_count', 'dq_issue_columns']].head(10))

df['anomaly'] = df['dq_issue_columns'].apply(lambda x: bool(x))

# ---------------------------------------
# Save result
# ---------------------------------------
output_path = r"C:\...\bets_with_dq_flags.csv"
df.to_csv(output_path, index=False)
print(f"\n✅ Data with DQ flags saved to:\n{output_path}")
df['dq_issue_columns'].value_counts()

#Task 2

In [ ]:
# Plot: Margin by Category with labels
plt.figure(figsize=(8, 5))
bars = plt.bar(margin_by_category['category'], margin_by_category['margin'], color='skyblue')

# Add labels on top of each bar
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2, height, f'{height:.2%}',  # format as percentage
             ha='center', va='bottom', fontsize=10)

plt.title('Margin by Category')
plt.ylabel('Margin (Return / Betting Amount)')
plt.grid(True, axis='y')
plt.tight_layout()

category_plot_path = r"C:\...\margin_by_category.png"
plt.savefig(category_plot_path)

plt.show()


In [ ]:

# 2. By price bin
bins = [1, 2, 5, 10, 20, float('inf')]
labels = ['(1,2]', '(2,5]', '(5,10]', '(10,20]', '(20+)']
clean_df['price_bin'] = pd.cut(clean_df['price'], bins=bins, labels=labels, right=True)
margin_by_price_bin = clean_df.groupby('price_bin').apply(
    lambda g: (g['return_for_entain'].sum() / g['betting_amount'].sum())
).rename('margin').reset_index()




In [ ]:
# Plot: Margin by Price Bin with labels
plt.figure(figsize=(8, 5))
bars = plt.bar(margin_by_price_bin['price_bin'], margin_by_price_bin['margin'], color='skyblue')

# Add labels on top of each bar
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2, height, f'{height:.2%}',
             ha='center', va='bottom', fontsize=10)

plt.title('Margin by Price Bin')
plt.ylabel('Margin (Return / Betting Amount)')
plt.xlabel('Price Bin')
plt.grid(True, axis='y')
plt.tight_layout()
price_plot_path = r"C:\...\margin_by_price_bin.png"
plt.savefig(price_plot_path)
plt.show()


In [ ]:
# 3. By stake_type
margin_by_stake_type = clean_df.groupby('stake_type').apply(
    lambda g: (g['return_for_entain'].sum() / g['betting_amount'].sum())
).rename('margin').reset_index()

In [ ]:
# Plot: Margin by Stake Type with labels
plt.figure(figsize=(8, 5))
bars = plt.bar(margin_by_stake_type['stake_type'], margin_by_stake_type['margin'], color='skyblue')

# Add percentage labels on top of each bar
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2, height, f'{height:.2%}',
             ha='center', va='bottom', fontsize=10)

plt.title('Margin by Stake Type')
plt.ylabel('Margin (Return / Betting Amount)')
plt.grid(True, axis='y')
plt.tight_layout()
stake_plot_path = r"C:\....\margin_by_stake_type.png"
plt.savefig(stake_plot_path)
plt.show()


In [ ]:

# --- Q3: Aggregate at customer level and plot ---
customer_summary = clean_df.groupby('customer_id').agg(
    total_return=('return_for_entain', 'sum'),
    avg_bet_amt=('betting_amount', 'mean')
).reset_index()

# Plot
plt.figure(figsize=(10, 6))
plt.scatter(customer_summary['avg_bet_amt'], customer_summary['total_return'], alpha=0.5)
plt.title('Customer Level: Return for Entain vs. Average Betting Amount')
plt.xlabel('Average Betting Amount (AUD)')
plt.ylabel('Total Return for Entain (AUD)')
plt.grid(True)
plt.tight_layout()
plot_path = r"C:\...\customer_return_vs_bet.png"
plt.savefig(plot_path)


In [ ]:
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import numpy as np

# --- Q3: Aggregate at customer level ---
customer_summary = clean_df.groupby('customer_id').agg(
    total_return=('return_for_entain', 'sum'),
    avg_bet_amt=('betting_amount', 'mean')
).reset_index()

# Prepare variables for regression
X = customer_summary[['avg_bet_amt']]  # must be 2D
y = customer_summary['total_return']

# Fit linear regression model
model = LinearRegression()
model.fit(X, y)
y_pred = model.predict(X)

# Calculate R²
r2 = r2_score(y, y_pred)
print(f"R²: {r2:.4f}")

# --- Plot with regression line ---
plt.figure(figsize=(10, 6))
plt.scatter(customer_summary['avg_bet_amt'], customer_summary['total_return'], alpha=0.5, label='Customers')
plt.plot(customer_summary['avg_bet_amt'], y_pred, color='red', label=f'Linear Fit\nR²={r2:.2f}')
plt.title('Customer Level: Return for Entain vs. Average Betting Amount')
plt.xlabel('Average Betting Amount (AUD)')
plt.ylabel('Total Return for Entain (AUD)')
plt.legend()
plt.grid(True)
plt.tight_layout()

# Save plot
plot_path = r"C:\...\customer_return_vs_bet.png"
plt.savefig(plot_path)
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# Filter out non-positive values (log is undefined for 0 or negative)
filtered = customer_summary[(customer_summary['avg_bet_amt'] > 0) & (customer_summary['total_return'] > 0)].copy()

# Log transform
filtered['log_avg_bet_amt'] = np.log(filtered['avg_bet_amt'])
filtered['log_total_return'] = np.log(filtered['total_return'])

# Fit linear model on log-log
X_log = filtered[['log_avg_bet_amt']]
y_log = filtered['log_total_return']
log_model = LinearRegression()
log_model.fit(X_log, y_log)
y_log_pred = log_model.predict(X_log)

# Calculate R² in log space
r2_log = r2_score(y_log, y_log_pred)
print(f"Log-Log R²: {r2_log:.4f}")

# Plot
plt.figure(figsize=(10, 6))
plt.scatter(filtered['log_avg_bet_amt'], filtered['log_total_return'], alpha=0.5, label='Log-Transformed Customers')
plt.plot(filtered['log_avg_bet_amt'], y_log_pred, color='red', label=f'Log-Log Fit\nR²={r2_log:.2f}')
plt.title('Log-Log Model: log(Total Return) vs. log(Avg Betting Amount)')
plt.xlabel('log(Average Betting Amount)')
plt.ylabel('log(Total Return for Entain)')
plt.legend()
plt.grid(True)
plt.tight_layout()

# Save plot
loglog_plot_path = r"C:\...\loglog_customer_return_vs_bet.png"
plt.savefig(loglog_plot_path)
plt


In [ ]:
# Calculate residuals
filtered['residual'] = filtered['log_total_return'] - y_log_pred
std_resid = filtered['residual'].std()

# Mark outliers as points with large residuals
filtered['is_outlier'] = filtered['residual'].abs() > 2 * std_resid


In [ ]:
plt.figure(figsize=(10, 6))

# Plot normal points
normal = filtered[~filtered['is_outlier']]
plt.scatter(normal['log_avg_bet_amt'], normal['log_total_return'], alpha=0.5, label='Customers')

# Plot outliers
outliers = filtered[filtered['is_outlier']]
plt.scatter(outliers['log_avg_bet_amt'], outliers['log_total_return'], color='red', alpha=0.8, label='Outliers')

# Regression line
plt.plot(filtered['log_avg_bet_amt'], y_log_pred, color='blue', label=f'Log-Log Fit\nR²={r2_log:.2f}')

plt.title('Log-Log Model: Total Return vs Avg Betting Amount (with Outliers)')
plt.xlabel('log(Average Betting Amount)')
plt.ylabel('log(Total Return for Entain)')
plt.legend()
plt.grid(True)
plt.tight_layout()

# Save
highlighted_plot_path = r"C:\...\loglog_outliers_highlighted.png"
plt.savefig(highlighted_plot_path)
plt.show()
print(f"✅ Plot saved to: {highlighted_plot_path}")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# Step 1: Filter and transform
filtered = customer_summary[(customer_summary['avg_bet_amt'] > 0) & (customer_summary['total_return'] > 0)].copy()
filtered['log_avg_bet_amt'] = np.log(filtered['avg_bet_amt'])
filtered['log_total_return'] = np.log(filtered['total_return'])

# Step 2: Initial fit
model = LinearRegression()
X_log = filtered[['log_avg_bet_amt']]
y_log = filtered['log_total_return']
model.fit(X_log, y_log)
y_pred = model.predict(X_log)

# Step 3: Identify outliers (residual > 2 std)
filtered['residual'] = y_log - y_pred
std_resid = filtered['residual'].std()
filtered['is_outlier'] = filtered['residual'].abs() > 2 * std_resid

# Step 4: Remove outliers
filtered_clean = filtered[~filtered['is_outlier']].copy()

# Step 5: Refit log-log model without outliers
X_clean = filtered_clean[['log_avg_bet_amt']]
y_clean = filtered_clean['log_total_return']
model_clean = LinearRegression()
model_clean.fit(X_clean, y_clean)
y_clean_pred = model_clean.predict(X_clean)

# Step 6: Calculate new R²
r2_clean = r2_score(y_clean, y_clean_pred)
print(f"✅ New Log-Log R² (no outliers): {r2_clean:.4f}")

# Step 7: Plot
plt.figure(figsize=(10, 6))
plt.scatter(filtered_clean['log_avg_bet_amt'], filtered_clean['log_total_return'], alpha=0.5, label='Customers (Clean)')
plt.plot(filtered_clean['log_avg_bet_amt'], y_clean_pred, color='green', label=f'Clean Log-Log Fit\nR²={r2_clean:.2f}')
plt.title('Log-Log Model (Outliers Removed)')
plt.xlabel('log(Average Betting Amount)')
plt.ylabel('log(Total Return for Entain)')
plt.legend()
plt.grid(True)
plt.tight_layout()

# Save the cleaned fit plot
clean_fit_path = r"C:\...\loglog_clean_fit.png"
plt.savefig(clean_fit_path)
plt.show()
